# 02 · 파형을 28개의 측정값으로 바꾼다

5분 FHR·TOCO 파형에서 수준, 변동, 가속·감속, 수축과 두 신호의 관계를 계산한다. 이 단계는 모델을 학습하지 않는다. 이후 Cat28 모델에 넣을 28개 특징을 만들고, 무엇이 측정되고 무엇이 측정되지 않는지 확인한다. Cat18은 그중 앞 18개를 사용한다.

**흐름:** 저장된 구간 읽기 → 결측 보간과 평활 → 특징 계산 → 품질·파형·기록 주석 비교 → 학습 집합의 관계 확인 → 저장

**입력:** `00`의 `data/prepared/`와 `01`의 `data/results/segments.csv`, `protocol.json`  
**출력:** `data/results/`의 특징 표, CNN용 신호 배열, 측정 품질과 비교 결과

`03-요약된-파일/02-파형과-측정.ipynb`의 계산 정의와 특징 순서를 유지한다. 입력 대상과 산모 분할은 `04`의 00·01 결과를 따르므로 과거 실험의 코호트와 같다고 가정하지 않는다. 이 폴더에서 00 → 01 → 02 순서로 실행한다.

출력 기준은 `MIN_PUBLIC_MOTHERS = 5`다. 02의 특징 그룹·사이트 표와 07의 사이트별 표·그림은 서로 다른 산모가 이보다 적으면 숨긴다. 내부 CSV와 학습·평가 계산은 전체 자료를 유지한다. 이 기준은 표시를 제한하는 기본값이며 기관의 반출 기준을 대신하지 않는다.

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.signal import find_peaks, peak_widths
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

DATASET = Path("data/prepared")
RESULTS = Path("data/results")
FS = 0.5  # One sample every 2 seconds; 150 samples per segment.
MIN_PUBLIC_MOTHERS = 5  # Display rule only; never changes training or saved measurements.


### 출력에 남기는 오류 정보

오류는 고정된 분류와 건수로만 요약한다. `other_error`의 상세 원인은 내부 오류 파일에서 확인한다. 건수는 오류 내역 수이며, 같은 기록이 여러 번 포함될 수 있다.

In [ ]:
def issue_category(reason):
    # Match known phrases, but return ONLY fixed labels: never echo an exception.
    # Example: "FileNotFoundError: /private/DEMO.json" -> "missing_input".
    text = str(reason)
    groups = {
        "missing_input": ("FileNotFoundError", "No such file", "[Errno 2]"),
        "missing_dependency": ("No module named", "cannot import name", "ImportError"),
        "fhr_missing": ("FHR 결측률",),
        "invalid_wave": ("FHR과 TOCO", "측정 간격", "FHR은 0 이상", "신호 길이"),
        "invalid_bbox": ("Bbox",),
        "invalid_id": ("ID가", "산모 ID", "산모 목록", "행 정보 불일치", "구간 정보 불일치"),
        "invalid_labels": ("라벨이 올바르지", "라벨 수", "라벨·점수"),
        "no_usable_data": ("구간이 없습니다", "비었", "남은 산모가 없습니다"),
        "split_unavailable": ("분할", "양성·음성", "정상·비정상", "least populated class", "test_size"),
        "checkpoint_mismatch": ("체크포인트", "CNN 입력·설정"),
        "io_error": ("PermissionError", "Permission denied", "No space left", "OSError"),
        "invalid_value": ("ValueError", "TypeError", "유한", "could not convert", "ufunc"),
    }
    for category, phrases in groups.items():
        if any(phrase in text for phrase in phrases):
            return category
    return "other_error"

In [ ]:
def show_issue_counts(reasons):
    # Count issue entries, not unique people; one record may have several entries.
    categories = pd.Series([issue_category(reason) for reason in reasons], dtype="string")
    counts = categories.value_counts().rename_axis("category").to_frame("count")
    print("제외·오류 내역 수 =", len(categories))
    # Full messages, IDs, paths, and jobs stay in the internal CSV files.
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(counts)

## 1. 분할 표와 신호 배열을 읽는다

한 행은 선택 태아의 5분 구간이다. `signal_index`로 원본 배열의 행을 찾고, `split`은 01에서 정한 산모별 배정을 그대로 사용한다. 배열은 `(구간 수, 2, 150)`이며 채널 순서는 FHR, TOCO다.

표와 배열의 연결이 어긋나면 계산을 멈춘다. 분할이 불가능했던 데이터도 특징은 계산할 수 있지만, 학습 집합의 관계 그림은 만들지 않는다.

In [ ]:
# IDs are strings; "00DEMO527183" is a fictional example with leading zeros.
id_types = {key: str for key in ("row_id", "record_id", "mother_id", "site")}
window_table = pd.read_csv(
    RESULTS / "segments.csv", dtype=id_types, keep_default_na=False,
)
# Records hold one row per fetus recording and its expert annotations.
record_table = pd.read_csv(DATASET / "records.csv", dtype=id_types)
signal_array = np.load(DATASET / "raw_signals.npy", mmap_mode="r")
protocol = json.loads((RESULTS / "protocol.json").read_text(encoding="utf-8"))

구간 표의 `signal_index`가 원본 배열의 같은 행을 가리키는지 먼저 확인한다. 이 연결이 깨지면 특징과 라벨을 맞출 수 없다.

In [ ]:
# Example: signal_index=3 points to signal_array[3], with shape (2, 150).

assert signal_array.shape == (len(window_table), 2, 150), "표와 신호 배열 크기가 다릅니다"
# Example: table indices [0, 1, 2] must match array rows 0, 1, 2.
assert np.array_equal(
    window_table["signal_index"], np.arange(len(window_table))
), "신호 인덱스 순서가 배열 행 순서와 다릅니다"

In [ ]:
print("segments =", len(window_table))
print("signal shape =", signal_array.shape)
print("split_ready =", protocol["split_ready"])

## 2. 결측을 채우고 평활 유무를 비교한다

특징 계산에서는 FHR·TOCO 모두 0을 보간 대상으로 삼는다. 먼저 7개 이하의 연속된 0을 채운다. 양옆에 값이 있으면 선형 보간하고, 한쪽만 있으면 그 값을 연장한다. 채울 값이 전혀 없으면 0으로 남긴다.

30초 평활은 15개 표본 안에서 0을 제외한 평균을 구한다. 평활 직전 0이었던 위치는 다시 0으로 돌려놓고, 마지막에 길이 제한 없이 보간한다. 비평활 조건은 중간의 평균 계산만 생략한다. 따라서 **두 조건 모두 마지막에는 보간 가능한 긴 결측도 채운다.**

CNN용 입력은 뒤에서 따로 만든다. FHR만 길이 제한 없이 보간하고 TOCO는 원래 값을 보존한다.

In [ ]:
def find_runs(mask, minimum=1):
    # Example: [False, True, True, False] -> [(1, 3)].
    # The end index is excluded, just like values[1:3].
    # False at both ends also detects a run touching index 0 or the last index.
    changes = np.diff(np.r_[False, mask, False].astype(int))
    starts = np.flatnonzero(changes == 1)
    ends = np.flatnonzero(changes == -1)

    intervals = []
    for start, end in zip(starts, ends):
        # Example: (1, 3) has length 2, so minimum=3 discards it.
        if end - start >= minimum:
            intervals.append((start, end))
    return intervals

In [ ]:
def impute_zeros(values, maximum):
    # Example: [140, 0, 0, 146] -> [140, 142, 144, 146] when maximum >= 2.
    values = np.asarray(values, dtype=float).copy()

    for start, end in find_runs(values == 0):
        # Example: a 3-sample zero run stays untouched when maximum=2.
        if end - start > maximum:
            continue

        left = values[start - 1] if start > 0 else np.nan
        right = values[end] if end < len(values) else np.nan

        # At an edge, copy the one known neighbor; all-zero input stays zero.
        if np.isnan(left) and np.isnan(right):
            continue
        if np.isnan(left):
            values[start:end] = right
        elif np.isnan(right):
            values[start:end] = left
        else:
            values[start:end] = np.linspace(left, right, end - start + 2)[1:-1]

    return values

짧은 0 구간을 보간한 뒤 15개 표본의 평균을 낸다. 평균을 낼 때 제외한 0의 위치는 마지막 보간 단계로 넘긴다.

In [ ]:
def preprocess(values, smooth):
    # int(15 * 0.5) = 7 samples: fill short gaps before smoothing.
    values = impute_zeros(values, maximum=int(15 * FS))

    if smooth:
        # 15 samples = 30 seconds. Divide each local sum by its nonzero count.
        # Example: local values [140, 0, 150] give (140 + 150) / 2 = 145.
        valid = values != 0
        kernel = np.ones(15)
        valid_count = np.convolve(valid.astype(float), kernel, mode="same")
        value_sum = np.convolve(values, kernel, mode="same")
        values = np.divide(
            value_sum, valid_count,
            out=np.zeros_like(values), where=valid_count > 0,
        )
        # Keep a still-missing position missing until the final interpolation.
        values[~valid] = 0

    # Fill every remaining gap that has at least one neighboring value.
    return impute_zeros(values, maximum=len(values))

In [ ]:
# Synthetic example only: interpolate two missing values between fictional endpoints.
print("imputed example =", impute_zeros([140, 0, 0, 146], maximum=2).tolist())

## 3. 기저선과 기저선 주변의 변동을 구한다

기저선은 양수 FHR의 평균에서 시작한다. 현재 기저선과의 차이가 15 bpm 미만인 값만 골라 평균을 갱신하고, 변화가 0.5 bpm 미만이거나 50회에 도달하면 끝낸다. 반환하는 마스크는 마지막 평균을 계산할 때 사용한 위치다.

변이도는 그 마스크가 3개 이상 연속된 곳에서 구한다. 상승·하강 방향이 바뀌는 극값을 찾고, 같은 연속 구간 안의 인접 극값 사이 절대차를 모아 평균한다. 평평한 차이는 건너뛰며 구간 끝점을 극값으로 추가하지 않는다. 극값 쌍이 없으면 0이다.

In [ ]:
def estimate_baseline(fhr):
    valid_fhr = fhr[fhr > 0]
    if len(valid_fhr) == 0:
        raise ValueError("기저선 계산에 사용할 FHR이 없습니다")

    # The original rule uses up to 600 seconds of positive samples.
    # A 150-sample segment fits entirely within this limit.
    baseline = valid_fhr[:int(600 * FS)].mean()

    # Example: [140, 142, 170] starts near 151; the 170-bpm point
    # lies more than 15 bpm away, so the next mean is (140 + 142) / 2 = 141.
    for _ in range(50):
        # Strictly less than 15: a point exactly 15 bpm away is excluded.
        baseline_mask = (np.abs(fhr - baseline) < 15) & (fhr > 0)
        updated_baseline = baseline
        if baseline_mask.any():
            updated_baseline = fhr[baseline_mask].mean()

        difference = abs(updated_baseline - baseline)
        baseline = updated_baseline
        if difference < 0.5:
            break

    # The mask describes the points used for the last update.
    return baseline, baseline_mask

In [ ]:
def baseline_variability(fhr, baseline_mask):
    amplitudes = []

    # Work within each retained run; never connect extrema across gaps.
    for start, end in find_runs(baseline_mask, minimum=3):
        values = fhr[start:end]
        previous_direction = 0
        extremes = []

        for index in range(1, len(values)):
            direction = np.sign(values[index] - values[index - 1])
            if direction == 0:
                continue
            # A rise followed by a fall marks the previous point as a peak.
            # Flat steps and run endpoints do not create an extremum.
            if previous_direction != 0 and direction != previous_direction:
                extremes.append(values[index - 1])
            previous_direction = direction

        # Example: [140, 145, 138, 143, 139] has extrema [145, 138, 143].
        # Its two amplitudes are 7 and 5 bpm; their mean is 6 bpm.
        if len(extremes) >= 2:
            amplitudes.extend(np.abs(np.diff(extremes)))

    if not amplitudes:
        return 0.0
    return float(np.mean(amplitudes))

In [ ]:
# Example: 170 bpm is too far from the updated baseline of 141 bpm.
example_fhr = np.array([140.0, 142.0, 170.0])
example_baseline, example_mask = estimate_baseline(example_fhr)
print("example baseline =", example_baseline)
print("samples used for baseline =", example_mask.tolist())

## 4. 사건과 두 신호의 관계를 측정한다

### 4.1. 수축과 감속 유형

수축은 TOCO의 봉우리로 근사한다. 봉우리 사이 간격은 60초 이상, prominence는 TOCO 표준편차의 절반 이상이어야 한다. 봉우리에서 prominence의 90%만큼 내려온 높이에서 폭을 재고, 45–120초인 것만 남긴다. 폭 조건은 반올림 전에 검사한다.

감속 후보는 기저선보다 15 bpm 이상 낮은 양수 FHR가 8개 이상 이어진 곳이다. 사건 길이는 표본 수 × 2초로 계산하므로 8개는 16초다. 감속 시작과 가장 가까운 수축 시작을 골라 아래 순서로 분류한다.

| 유형 | 코드의 조건 |
| --- | --- |
| late | 수축 시작 20초 후보다 늦고, 수축 종료 전인 감속 시작 |
| early | late가 아니며 길이가 180초 미만 |
| prolonged | late가 아니며 길이가 180–300초 |
| severe | late가 아니며 길이가 300초 초과 |
| variable | 시작부터 첫 최저점까지 30초 미만; 위 유형과 별도로 집계 |

이것은 원 코드의 근사 규칙이다. 이름만으로 임상 판독과 같다고 볼 수 없으며, 5분 구간에서는 severe 조건을 관측할 수 없다.

In [ ]:
def find_contractions(toco):
    if toco.std() == 0:
        return []

    # At FS=0.5, distance=30 samples means peaks at least 60 seconds apart.
    # Prominence must reach half the TOCO standard deviation.
    peaks, _ = find_peaks(
        toco, distance=int(60 * FS), prominence=toco.std() * 0.5,
    )
    if len(peaks) == 0:
        return []

    # Width is measured where the peak has fallen by 90% of its prominence.
    _, _, left_edges, right_edges = peak_widths(toco, peaks, rel_height=0.9)
    contractions = []
    for peak, left, right in zip(peaks, left_edges, right_edges):
        duration_seconds = (right - left) / FS
        # Check the unrounded width first, then store rounded indices.
        if 45 <= duration_seconds <= 120:
            contractions.append((int(round(left)), int(peak), int(round(right))))

    # Each tuple contains sample indices: (start, peak, end).
    return contractions

In [ ]:
def count_decelerations(fhr, decelerations, contractions):
    counts = dict(early=0, late=0, variable=0, severe=0, prolonged=0)

    for start, end in decelerations:
        is_late = False
        if contractions:
            # Use the contraction with the nearest onset, not the nearest peak.
            nearest = min(contractions, key=lambda contraction: abs(start - contraction[0]))
            onset, _, offset = nearest
            # Example: onset=40, start=51, offset=70 is late (40 + 10 < 51).
            # start=50 is not late: equality does not pass the strict test.
            is_late = onset + 20 * FS < start < offset

        # Example: 90 samples at 2 seconds each = 180 seconds, prolonged.
        duration_seconds = (end - start) / FS
        if is_late:
            kind = "late"
        elif duration_seconds < 180:
            kind = "early"
        elif duration_seconds <= 300:
            kind = "prolonged"
        else:
            kind = "severe"
        counts[kind] += 1

        # A deceleration can be both early and variable; do not sum all five counts.
        samples_to_minimum = np.argmin(fhr[start:end])
        # At FS=0.5, the first minimum must occur before sample 15.
        if samples_to_minimum < 30 * FS:
            counts["variable"] += 1

    return counts

### 4.2. FHR·TOCO 지연 상관

동시점 상관과 ±60초 범위의 최소 상관을 구한다. 지연이 양수이면 뒤의 FHR와 앞의 TOCO를 비교한다. 가장 작은 상관을 고르는 것이며 절댓값이 가장 큰 상관을 고르는 것은 아니다.

두 값이 같으면 기존 선택을 유지한다. 따라서 최소 상관에 동점이 있으면 0초 지연이 우선이고, 0초가 최소가 아니면 먼저 검사한 지연이 남는다. 상수 신호 등으로 상관을 계산할 수 없을 때 반환하는 0은 원 코드의 대체값이다.

In [ ]:
def correlation(first, second):
    # A constant channel has no defined Pearson correlation here; 0 is
    # the original algorithm's placeholder, not evidence of no relation.
    if len(first) < 2 or first.std() == 0 or second.std() == 0:
        return 0.0
    return float(np.corrcoef(first, second)[0, 1])

In [ ]:
def correlation_features(fhr, toco):
    simultaneous = correlation(fhr, toco)
    minimum = simultaneous
    selected_lag = 0

    for lag in range(-30, 31):
        if lag == 0:
            continue

        # Example: lag=5 pairs fhr[5:] with toco[:-5], a 10-second FHR delay.
        if lag > 0:
            shifted_fhr = fhr[lag:]
            shifted_toco = toco[:-lag]
        else:
            # Example: lag=-5 pairs fhr[:-5] with toco[5:].
            shifted_fhr = fhr[:lag]
            shifted_toco = toco[-lag:]

        value = correlation(shifted_fhr, shifted_toco)
        # Replace the current choice only when smaller; ties keep lag 0
        # or the first previously visited lag.
        if value < minimum:
            minimum = value
            selected_lag = lag

    return dict(
        ft_corr0=simultaneous,
        ft_corr_min=minimum,
        ft_corr_min_lag_s=selected_lag / FS,
    )

### 4.3. 한 구간을 Cat28 순서로 요약한다

아래 순서는 이후 모델이 사용할 열 순서다. 모든 특징은 전처리한 FHR·TOCO에서 계산하며, 전문가의 기록별 주석은 입력 특징에 넣지 않는다.

가속은 기저선보다 15 bpm 이상 높은 양수 FHR가 8개 이상 이어져야 한다. 전체 길이 600초 미만, 시작부터 첫 최고점까지 30초 미만이라는 조건도 적용한다. `stv`는 인접한 2초 표본 사이 절대차의 평균이며, 박동별 임상 STV와 다르다.

FHR 값의 단위는 bpm, TOCO는 원본 장비의 척도다. 표준편차는 `ddof=0`으로 계산한다. `hist_mode`는 24칸 히스토그램 중 첫 최빈 칸의 중앙값이다. 상수 파형에서도 칸 중앙을 사용하므로 원래 상수와 조금 다를 수 있다.

In [ ]:
# Column order is part of the model input: Cat18 is exactly positions 0–17.
CAT28 = [
    # FHR level, spread, and threshold fractions.
    "fhr_mean", "fhr_min", "fhr_max", "fhr_sd", "stv", "brady_frac", "tachy_frac",
    # Detected events, then TOCO and FHR–TOCO coupling.
    "n_decel", "decel_max_depth", "decel_time_frac", "n_accel",
    "toco_mean", "toco_max", "toco_sd", "n_contractions",
    "ft_corr0", "ft_corr_min", "ft_corr_min_lag_s",
    # Baseline, deceleration types, and histogram summaries.
    "figo_baseline", "figo_baseline_var",
    "n_early_decel", "n_late_decel", "n_variable_decel",
    "n_severe_decel", "n_prolonged_decel",
    "hist_width", "hist_median", "hist_mode",
]
CAT18 = CAT28[:18]

In [ ]:
def detect_feature_events(fhr, toco, baseline):
    # ceil(15 * 0.5) = 8 samples, counted as 16 seconds by the event rule.
    minimum_samples = int(np.ceil(15 * FS))
    # Example: baseline=140, FHR=125 for 8 samples -> one 16-second
    # deceleration. Zero values never count as decelerations.
    deceleration_mask = (fhr <= baseline - 15) & (fhr > 0)
    decelerations = find_runs(deceleration_mask, minimum=minimum_samples)

    # Example: baseline=140 requires FHR >=155 for at least 8 samples.
    acceleration_mask = (fhr >= baseline + 15) & (fhr > 0)
    accelerations = []
    for start, end in find_runs(acceleration_mask, minimum=minimum_samples):
        samples_to_peak = np.argmax(fhr[start:end])
        # The first peak must be before sample 15 (30 seconds).
        if end - start < 600 * FS and samples_to_peak < 30 * FS:
            accelerations.append((start, end))

    contractions = find_contractions(toco)
    return decelerations, accelerations, contractions

In [ ]:
def extract_features(fhr, toco):
    baseline, baseline_mask = estimate_baseline(fhr)
    decelerations, accelerations, contractions = detect_feature_events(
        fhr, toco, baseline
    )

    deceleration_counts = count_decelerations(fhr, decelerations, contractions)
    depths = [baseline - fhr[start:end].min() for start, end in decelerations]
    deceleration_samples = sum(end - start for start, end in decelerations)

    # Hist mode is the midpoint of the first most-populated bin.
    histogram, edges = np.histogram(fhr, bins=24)
    most_frequent_bin = histogram.argmax()
    mode = (edges[most_frequent_bin] + edges[most_frequent_bin + 1]) / 2

    values = dict(
        fhr_mean=fhr.mean(),
        fhr_min=fhr.min(),
        fhr_max=fhr.max(),
        fhr_sd=fhr.std(),
        # Example: [140, 144, 142] -> mean adjacent difference (4+2)/2=3.
        stv=np.abs(np.diff(fhr)).mean(),
        brady_frac=np.mean(fhr < 110),
        tachy_frac=np.mean(fhr > 160),
        n_decel=len(decelerations),
        decel_max_depth=max(depths, default=0),
        # Example: one 10-sample event / 150 samples -> 1/15.
        decel_time_frac=deceleration_samples / len(fhr),
        n_accel=len(accelerations),
        toco_mean=toco.mean(),
        toco_max=toco.max(),
        toco_sd=toco.std(),
        n_contractions=len(contractions),
        figo_baseline=baseline,
        figo_baseline_var=baseline_variability(fhr, baseline_mask),
        hist_width=fhr.max() - fhr.min(),
        hist_median=np.median(fhr),
        hist_mode=mode,
    )
    # Example: counts["early"]=1 becomes values["n_early_decel"]=1.
    for kind, count in deceleration_counts.items():
        values[f"n_{kind}_decel"] = count
    values.update(correlation_features(fhr, toco))

    # Example: list(features) starts with ["fhr_mean", "fhr_min", "fhr_max"].
    return {feature: float(values[feature]) for feature in CAT28}

In [ ]:
# A simple example passed directly to extraction, without another smoothing step.
# FHR is 140 bpm, except for 10 samples at 120 bpm; TOCO is constant zero.
example_fhr = np.full(150, 140.0)
example_fhr[50:60] = 120
example_toco = np.zeros(150)
example_features = extract_features(example_fhr, example_toco)

# Expected: one 20-second deceleration, depth 20 bpm, both early and variable.
print("feature count =", len(example_features))
for feature in (
    "figo_baseline", "n_decel", "decel_max_depth", "decel_time_frac",
    "n_early_decel", "n_variable_decel", "n_severe_decel",
):
    print(feature, "=", example_features[feature])

## 5. 같은 구간에서 두 특징 표를 만든다

각 구간에서 30초 평활과 비평활 특징을 모두 계산한다. 두 조건이 모두 유한한 28개 값을 만들 때만 함께 추가한다. 한 구간에서 예상되는 입력 오류가 나면 사유를 남기고 다음 구간을 처리한다.

CNN용 배열은 원본과 같은 크기로 만들고 `signal_index` 위치에 저장한다. 제외된 구간의 자리는 0으로 남지만 특징 표에는 그 행이 없다. 이후 CNN도 **특징 표의 `signal_index`로 신호를 선택**해야 한다. 특징 표의 행 번호로 배열을 찾거나 인덱스를 다시 매기면 안 된다.

In [ ]:
def extract_one_segment(row):
    signal_index = row["signal_index"]
    fhr, toco = signal_array[signal_index].astype(float)
    feature_rows = []
    elapsed_seconds = []

    # Both arms use the same row ID and label. Keep them together only
    # when both calculations succeed: arms[0]=smooth30, arms[1]=smooth0.
    for smooth in (True, False):
        started = time.perf_counter()
        processed_fhr = preprocess(fhr, smooth=smooth)
        processed_toco = preprocess(toco, smooth=smooth)
        features = extract_features(processed_fhr, processed_toco)
        elapsed_seconds.append(time.perf_counter() - started)

        if not np.isfinite(list(features.values())).all():
            raise ValueError("특징 계산 결과에 유한하지 않은 값이 있습니다")
        feature_rows.append(dict(row, **features))

    # CNN input: fill FHR gaps, but keep TOCO exactly as recorded.
    cnn_wave = np.stack([impute_zeros(fhr, maximum=len(fhr)), toco])
    return feature_rows[0], feature_rows[1], elapsed_seconds, cnn_wave

한 구간의 두 특징 행을 모두 만들 수 있을 때만 최종 목록에 넣는다. 구간 오류는 ID와 사유를 남기고 다음 구간으로 넘어간다.

In [ ]:
smoothed_rows = []
unsmoothed_rows = []
extraction_times = []
exclusions = []
cnn_signals = np.zeros_like(signal_array, dtype="float32")

for processed, row in enumerate(window_table.to_dict("records"), start=1):
    try:
        smoothed_row, unsmoothed_row, elapsed_seconds, cnn_wave = extract_one_segment(row)
        signal_index = row["signal_index"]

        # Example: signal_index=2 writes cnn_signals[2], even if index 1
        # failed and has no feature-table row.
        cnn_signals[signal_index] = cnn_wave
        smoothed_rows.append(smoothed_row)
        unsmoothed_rows.append(unsmoothed_row)
        extraction_times.append(elapsed_seconds)

    except (ValueError, TypeError, IndexError) as error:
        exclusions.append(dict(row_id=row["row_id"], reason=str(error)))

    if processed % 1000 == 0:
        print(processed, "/", len(window_table), "segments processed", flush=True)

In [ ]:
# The same retained row IDs and Cat28 order appear in both arms.
# Keep headers even when no segment survives.
feature_columns = list(window_table.columns) + CAT28
feature_table = pd.DataFrame(smoothed_rows, columns=feature_columns)
unsmoothed_table = pd.DataFrame(unsmoothed_rows, columns=feature_columns)
exclusion_table = pd.DataFrame(exclusions, columns=["row_id", "reason"])

In [ ]:
print("input segments =", len(window_table))
print("retained segments =", len(feature_table))
print("excluded segments =", len(exclusion_table))
print("Cat28 shape =", feature_table[CAT28].shape)
print("Cat18 shape =", feature_table[CAT18].shape)
print("CNN signal shape =", cnn_signals.shape)

if feature_table.empty:
    print("추출 가능한 구간이 없습니다. 후속 모델 실험을 진행할 수 없습니다.")

## 6. 측정 품질과 실제 파형을 확인한다

### 6.1. 어떤 특징이 상수인가

평활 특징 표에서 각 특징의 고유값 수, 0 비율, 최솟값·최댓값을 센다. 분모는 추출에 성공한 전체 구간 수다. `n_severe_decel`의 0은 5분보다 긴 사건을 이 구간 안에서 셀 수 없다는 뜻이며, 실제 severe 사건의 부재를 입증하지 않는다.

시간은 구간별 전처리와 28개 특징 계산에 걸린 평균이다. 파일 읽기·저장과 CNN 입력 생성 시간은 포함하지 않는다.

In [ ]:
quality_rows = []
for feature in CAT28:
    values = feature_table[feature]
    # Example: [0, 0, 1, 1] -> unique=2 and zero_fraction=0.5.
    unique_count = values.nunique()
    status = "variable"
    if unique_count <= 1:
        status = "constant"
    # A 5-minute window cannot contain a >300-second event.
    if feature == "n_severe_decel":
        status = "unobservable_in_5min"
    if feature_table.empty:
        status = "unavailable"

    quality_rows.append(dict(
        feature=feature,
        unique=unique_count,
        zero_fraction=values.eq(0).mean(),
        min=values.min(),
        max=values.max(),
        status=status,
    ))
quality_table = pd.DataFrame(quality_rows)

display(quality_table)

In [ ]:
timing_summary = dict(windows=len(feature_table), mean_ms_smooth30=None, mean_ms_smooth0=None)
if extraction_times:
    # Each pair contains [smooth30 seconds, smooth0 seconds].
    # Multiply the per-arm mean by 1000 to report milliseconds.
    mean_milliseconds = np.mean(extraction_times, axis=0) * 1000
    timing_summary["mean_ms_smooth30"] = float(mean_milliseconds[0])
    timing_summary["mean_ms_smooth0"] = float(mean_milliseconds[1])

print("extraction timing =", timing_summary)

In [ ]:
# Each bar uses retained segments as its denominator.
if not feature_table.empty:
    quality_table.set_index("feature")["zero_fraction"].plot.barh(
        figsize=(8, 8), title="Zero fraction by feature",
    )
    plt.xlabel("Fraction of retained segments")
    plt.tight_layout()
    plt.show()

## 7. 기록 단위의 주석과 비교한다

한 기록의 남은 구간에서 기저선·변이도는 평균, 사건 수는 합계로 모은다. 이를 00이 저장한 기록별 주석과 연결해 Spearman 상관을 구한다. 기저선은 평균 절대차(MAE, bpm), 사건은 주석상 존재 여부에 대한 AUROC도 계산한다. 가속 주석은 값 1만, 감속 주석은 양수를 존재로 본다.

표의 `records`는 해당 비교에 사용할 수 있는 기록 수다. `missing_records`에는 주석이 없거나 추출된 구간이 없어 비교할 수 없는 기록이 포함된다. 상수 값의 상관과 한 클래스뿐인 AUROC는 결측으로 남긴다.

이 비교는 모든 분할을 합친 측정 점검이다. 기록 주석을 구간별 정답으로 복제하지 않으며, 이 결과로 모델이나 특징을 선택하지 않는다. 제외된 구간이 있으면 사건 합계는 원래 기록 전체를 포괄하지 못한다.

In [ ]:
# Example: a record with segment baselines [140, 150] contributes
# figo_baseline=145; event counts [1, 2] contribute a sum of 3.
annotation_pairs = [
    ("BaseLine", "figo_baseline", "mean"),
    ("Baseline_Variability", "figo_baseline_var", "mean"),
    ("Acceleration", "n_accel", "sum"),
    ("Early_deceleration", "n_early_decel", "sum"),
    ("Late_deceleration", "n_late_decel", "sum"),
    ("Variable_deceleration", "n_variable_decel", "sum"),
    ("Prolonged_deceleration", "n_prolonged_decel", "sum"),
]

In [ ]:
def compare_annotation(annotation, feature, aggregation):
    annotation_column = f"label_{annotation}"
    record_features = feature_table.groupby("record_id")[feature].agg(aggregation)
    joined = record_table[["record_id", annotation_column]].join(
        record_features, on="record_id",
    ).dropna()
    # Keep records with both the annotation and an extracted aggregate.
    # missing_records also counts records with no successful segments.
    annotated_values = joined[annotation_column]
    measured_values = joined[feature]

    rank_correlation = np.nan
    # Rank correlation needs variation on both sides; otherwise use NaN.
    if annotated_values.nunique() > 1 and measured_values.nunique() > 1:
        rank_correlation = spearmanr(annotated_values, measured_values).statistic

    result = dict(
        annotation=annotation,
        feature=feature,
        records=len(joined),
        missing_records=len(record_table) - len(joined),
        spearman=rank_correlation,
    )
    if annotation == "BaseLine":
        result["MAE_bpm"] = float(np.abs(measured_values - annotated_values).mean())

    if aggregation == "sum":
        # Acceleration annotation means present only at 1; decelerations
        # mean present for any positive annotation value.
        if annotation == "Acceleration":
            event_present = annotated_values.eq(1)
        else:
            event_present = annotated_values.gt(0)

        result["positive_records"] = int(event_present.sum())
        result["AUROC_presence"] = np.nan
        # AUROC is undefined when every record has the same presence label.
        if event_present.nunique() == 2:
            result["AUROC_presence"] = roc_auc_score(event_present, measured_values)

    return result

In [ ]:
comparison_rows = []
for annotation, feature, aggregation in annotation_pairs:
    comparison_rows.append(compare_annotation(annotation, feature, aggregation))
annotation_comparison = pd.DataFrame(comparison_rows)

display(annotation_comparison)

## 8. 학습 집합의 특징·라벨 관계와 사이트 구성을 확인한다

### 8.1. 학습 집합에서만 관계를 그린다

특징값을 묶고, 각 묶음에서 평균 특징값과 비정상 구간 비율을 계산한다. 고유값이 5개 이하면 값별로 묶는다. 그 외에 0이 절반 이상이면 0을 따로 두고 나머지를 최대 4개 분위수로, 그렇지 않으면 최대 5개 분위수로 나눈다. 같은 경계는 제거한다.

비정상 비율의 분모는 그 묶음의 구간 수다. 다른 특징을 고정한 효과나 인과관계가 아니며, 같은 산모의 구간도 각각 센다. 검증·시험 데이터는 이 곡선에 넣지 않는다.

In [ ]:
def feature_groups(values):
    # Example: counts [0, 1, 2] form three groups, not quantile bins.
    if values.nunique() <= 5:
        return values

    # Example: 6 zeros / 10 values -> one "zero" group; divide the
    # four nonzero values into up to four quantile groups.
    if values.eq(0).mean() >= 0.5:
        groups = pd.Series("zero", index=values.index)
        nonzero = values.ne(0)
        groups.loc[nonzero] = pd.qcut(
            values[nonzero], 4, duplicates="drop",
        ).astype(str)
        return groups

    return pd.qcut(values, 5, duplicates="drop")

In [ ]:
# Only the train split contributes to these label-response curves.
training = feature_table.loc[feature_table["split"].eq("train")]
response_rows = []
response_mothers = []

if protocol["split_ready"] and not training.empty:
    for feature in CAT28:
        groups = feature_groups(training[feature])

        # Example: 2 abnormal / 5 windows -> abnormal=0.4.
        # value is that group's mean feature value, not a bin boundary.
        grouped = training.groupby(groups, observed=True).agg(
            value=(feature, "mean"),
            abnormal=("target", "mean"),
            windows=("target", "size"),
            mothers=("mother_id", "nunique"),
        ).sort_values("value")
        response_mothers.extend(grouped["mothers"].tolist())
        response_rows.append(grouped.drop(columns="mothers").reset_index(drop=True).assign(feature=feature))
else:
    print("학습 관계 미산출: 분할이 준비되지 않았거나 남은 학습 구간이 없습니다.")

response_table = pd.DataFrame(columns=["value", "abnormal", "windows", "feature"])
if response_rows:
    response_table = pd.concat(response_rows, ignore_index=True)


In [ ]:
# Example: 100 windows from 2 mothers are still below a 5-mother display limit.
response_support = response_table.assign(mothers=response_mothers)
public_responses = response_support.loc[response_support["mothers"].ge(MIN_PUBLIC_MOTHERS)].copy()
print("공개 최소 산모 수 =", MIN_PUBLIC_MOTHERS)
print("숨긴 특징 그룹 수 =", len(response_table) - len(public_responses))
# Show every permitted feature/bin; do not truncate to the first five rows.
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(public_responses)

In [ ]:
if not public_responses.empty:
    figure, axes = plt.subplots(7, 4, figsize=(14, 17))
    # The dashed reference is the abnormal fraction of all train windows.
    training_prevalence = training["target"].mean()

    for axis, feature in zip(axes.flat, CAT28):
        # The plot uses the same permitted rows as the exported table.
        response = public_responses.loc[public_responses["feature"].eq(feature)]
        axis.plot(response["value"], response["abnormal"], "o-")
        axis.axhline(training_prevalence, color="gray", linestyle="--")
        axis.set(title=feature, ylim=(0, 1))

    figure.suptitle("Training-only abnormality by feature; dashed = training prevalence")
    figure.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()


### 8.2. 사이트별 표본 구성을 센다

추출에 성공한 전체 구간에서 사이트별 산모 수, 구간 수, 비정상 비율과 원본 FHR 결측률의 구간 평균을 구한다. 이는 표본 구성의 차이를 보여주며 사이트별 모델 성능은 아니다. 사이트 코드를 확인되지 않은 병원 이름으로 바꾸지 않는다.

In [ ]:
# Example: two records from one mother at site "DEMO_SITE" count as one mother.
# prevalence and missing_fraction are means over retained segments.
site_summary = feature_table.groupby("site").agg(
    mothers=("mother_id", "nunique"),
    windows=("target", "size"),
    prevalence=("target", "mean"),
    missing_fraction=("missing_fraction", "mean"),
)
public_sites = site_summary.loc[site_summary["mothers"].ge(MIN_PUBLIC_MOTHERS)]
print("숨긴 사이트 수 =", len(site_summary) - len(public_sites))
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(public_sites)


## 9. 다음 실험이 읽을 결과를 저장한다

`data/results/`의 기존 분할 표는 유지하고 아래 파일을 갱신한다. 기존 실험의 특징 열·순서와 결과 파일명을 보존하며, 구간별 오류는 이 단계의 `02_exclusions.csv`로 모은다.

| 파일 | 내용 |
| --- | --- |
| `features.csv` | 원래 구간 정보·분할과 30초 평활 Cat28 |
| `features_unsmoothed.csv` | 같은 구간·열 순서의 비평활 Cat28 |
| `cnn_signals.npy` | FHR만 보간한 배열; 원래 `signal_index`로 선택 |
| `feature_quality.csv` | 평활 특징의 고유값 수·0 비율·범위·상태 |
| `extraction_timing.json` | 성공 구간 수와 두 조건의 평균 계산 시간 |
| `02_annotation_comparison.csv` | 기록 주석과 집계 특징의 비교 |
| `02_train_responses.csv` | 학습 집합에서 묶은 특징값·비정상 비율·구간 수 |
| `02_site_composition.csv` | 사이트별 표본 구성 |
| `02_exclusions.csv` | 특징 추출에서 제외한 구간 ID와 사유 |

남은 구간이 없으면 특징 표를 열 이름만 있는 빈 표로 갱신한다. 학습 관계를 만들 수 없을 때도 해당 표를 비워 이전 실행의 결과가 남지 않게 한다. 빈 표는 후속 실험을 진행할 수 있다는 뜻이 아니다. 00이나 01을 다시 실행했다면 이 단계와 후속 분석도 다시 실행해야 한다.

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)

tables = {
    "features.csv": feature_table,
    "features_unsmoothed.csv": unsmoothed_table,
    "feature_quality.csv": quality_table,
    "02_annotation_comparison.csv": annotation_comparison,
    "02_train_responses.csv": response_table,
    "02_exclusions.csv": exclusion_table,
}
# Each table is regenerated from this run; empty tables keep headers
# so previous-run results cannot be mistaken for current results.
for filename, table in tables.items():
    table.to_csv(RESULTS / filename, index=False)

In [ ]:
# site_summary uses the site code as its index; keep it in the CSV.
site_summary.to_csv(RESULTS / "02_site_composition.csv")
np.save(RESULTS / "cnn_signals.npy", cnn_signals)
(RESULTS / "extraction_timing.json").write_text(
    json.dumps(timing_summary, ensure_ascii=False, indent=2), encoding="utf-8",
)

print("results directory =", RESULTS)
print("saved feature rows =", len(feature_table))
print("saved exclusions =", len(exclusion_table))

## 측정과 판정을 구별한다

28개 특징을 계산했다는 사실이 28개 모두의 임상 타당성을 뜻하지는 않는다. 5분 구간의 severe 수, 상수 신호의 상관 대체값, 2초 차이로 만든 STV는 각각의 계산 한계와 함께 읽어야 한다.

다음 실험은 이 표와 01의 산모 분할을 사용해 판독과의 관계 및 예측 성능을 평가한다. 여기서는 측정 정의를 고정하며, 전체 데이터의 품질 표나 기록 주석 비교를 보고 모델을 선택하지 않는다.

### 반출용 제외·오류 집계

개별 ID·경로·오류 전문을 제외한 분류별 건수다.

In [ ]:
show_issue_counts(exclusion_table["reason"])